# Technical Report Notebook

Este notebook percorre as etapas principais do projeto e exibe evid?ncias de qualidade do treinamento:

1. Carga e inspe??o de dados
2. Pr?-processamento temporal
3. Split cronol?gico + prepara??o de features
4. Avalia??o de checkpoint e gr?ficos de erro
5. Gera??o e visualiza??o do relat?rio t?cnico autom?tico

## 1) Ambiente e imports

Execute a c?lula abaixo para configurar o caminho do projeto e importar as fun??es principais.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ajusta o root quando o notebook for executado de dentro de docs/
ROOT = Path.cwd()
if ROOT.name == "docs":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.loader import load_data
from src.data.preprocessing import melt_maturities, add_temporal_features, normalize_prices, prepare_features
from src.data.splits import time_based_split
from src.inference.pipeline import load_model, preprocess_with_checkpoint, make_feature_matrix, predict_array
from src.eval.metrics import evaluate
from src.data.technical_report import TechnicalReportConfig, generate_technical_report

plt.style.use("seaborn-v0_8-whitegrid")
print(f"Project root: {ROOT}")

## 2) Carga dos dados

In [ ]:
df_raw = load_data(filename="train.xlsx", data_dir="DATASETS", parse_dates=True, date_col="Date")
print("Raw shape:", df_raw.shape)
print("Date range:", df_raw["Date"].min(), "->", df_raw["Date"].max())

display(df_raw.head())

## 3) Pipeline temporal (melt -> features -> normaliza??o)

In [ ]:
df_long = melt_maturities(df_raw, date_col="Date", value_name="price")
df_feat = add_temporal_features(
    df_long,
    date_col="Date",
    target_col="price",
    lags=(1, 5, 10),
    rolling_windows=(5, 20),
    dropna=True,
)
df_norm, scaler = normalize_prices(df_feat, price_col="price")

print("Long shape:", df_long.shape)
print("Feature shape:", df_norm.shape)
print("Scaler mean/std:", float(scaler.mean_[0]), float(scaler.scale_[0]))

display(df_norm.head())

In [ ]:
# Visual r?pido da s?rie m?dia de pre?o
mean_price_by_date = df_long.groupby("Date")["price"].mean().sort_index()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mean_price_by_date.index, mean_price_by_date.values)
ax.set_title("Average Price Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Mean price")
ax.tick_params(axis="x", rotation=35)
plt.show()

## 4) Split cronol?gico e features de treino/valida??o

In [ ]:
train_df, val_df, split_date = time_based_split(df_norm, val_fraction=0.2)
X_train, y_train, feature_cols = prepare_features(train_df, target_col="price_norm", return_feature_names=True)
X_val, y_val = prepare_features(val_df, feature_cols=feature_cols, target_col="price_norm")

print("Split date:", split_date)
print("Train rows:", len(train_df), "| Val rows:", len(val_df))
print("Feature columns:", len(feature_cols))
print("X_train:", X_train.shape, "X_val:", X_val.shape)

## 5) Qualidade do treinamento (hist?rico salvo)

In [ ]:
history_path = ROOT / "results" / "training_history.json"
if history_path.exists():
    history = json.loads(history_path.read_text(encoding="utf-8"))
    hist_df = pd.DataFrame([
        {
            "epoch": item["epoch"],
            "train_rmse": item["train_metrics"]["rmse"],
            "val_rmse": item["val_metrics"]["rmse"],
            "train_r2": item["train_metrics"]["r2"],
            "val_r2": item["val_metrics"]["r2"],
        }
        for item in history
    ])
    display(hist_df)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist_df["epoch"], hist_df["train_rmse"], marker="o", label="Train RMSE")
    axes[0].plot(hist_df["epoch"], hist_df["val_rmse"], marker="o", label="Val RMSE")
    axes[0].set_title("RMSE by epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("RMSE")
    axes[0].legend()

    axes[1].plot(hist_df["epoch"], hist_df["train_r2"], marker="o", label="Train R2")
    axes[1].plot(hist_df["epoch"], hist_df["val_r2"], marker="o", label="Val R2")
    axes[1].set_title("R2 by epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("R2")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("training_history.json not found")

## 6) Avalia??o de checkpoint (predi??o e erro)

In [ ]:
checkpoint_path = ROOT / "results" / "checkpoint.pt"
if checkpoint_path.exists():
    model, checkpoint = load_model(checkpoint_path)
    df_ckpt = preprocess_with_checkpoint(df_raw, checkpoint)
    _, val_ckpt, split_ckpt = time_based_split(df_ckpt, val_fraction=0.2)

    ckpt_feature_cols = list(checkpoint["feature_cols"])
    X_val_ckpt = make_feature_matrix(val_ckpt, ckpt_feature_cols)
    y_true = val_ckpt["price_norm"].to_numpy(dtype=np.float32)
    y_pred = predict_array(model, X_val_ckpt)

    metrics = evaluate(y_true, y_pred)
    print("Checkpoint split date:", split_ckpt)
    print("Validation rows:", len(val_ckpt))
    print("Metrics:", metrics)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    axes[0].scatter(y_true, y_pred, s=8, alpha=0.35)
    lo = float(min(y_true.min(), y_pred.min()))
    hi = float(max(y_true.max(), y_pred.max()))
    axes[0].plot([lo, hi], [lo, hi], "r--")
    axes[0].set_title("Predicted vs True (price_norm)")
    axes[0].set_xlabel("True")
    axes[0].set_ylabel("Predicted")

    residuals = y_true - y_pred
    axes[1].hist(residuals, bins=40)
    axes[1].set_title("Residual distribution")
    axes[1].set_xlabel("Residual")
    axes[1].set_ylabel("Frequency")

    plt.tight_layout()
    plt.show()
else:
    print("Checkpoint not found at", checkpoint_path)

## 7) Gerar relat?rio t?cnico autom?tico (`docs/technical_report.md`)

In [ ]:
config = TechnicalReportConfig(
    data_dir="DATASETS",
    train_filename="train.xlsx",
    checkpoint_path="results/checkpoint.pt",
    training_history_path="results/training_history.json",
    output_markdown="docs/technical_report.md",
    assets_dir="docs/assets/technical_report",
    benchmark_repeats=3,
    val_fraction=0.2,
)

report_path = generate_technical_report(config)
print("Report generated at:", report_path)

In [ ]:
# Mostra os gr?ficos gerados no relat?rio
from IPython.display import Image, display

assets = ROOT / "docs" / "assets" / "technical_report"
for name in [
    "benchmark_times.png",
    "data_profile.png",
    "training_quality.png",
    "prediction_scatter.png",
    "prediction_residuals.png",
]:
    path = assets / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))